# Бектест на прибыльность

**Шаг 7 плана.** Загрузка лучшей модели, бектест с комиссией 0.1%, варианты: торговать на каждом баре и с фильтром по confidence. Сравнение с наивной стратегией (target).

## 1. Загрузка модели и данных

**Важно:** Модель обучена в 09 с temporal split (8 дней train, 1 val, 1 test). Бектест только на **последнем дне** (test_date) — иначе in-sample завышает доходность.

In [17]:
import sys
import os
import numpy as np
import pandas as pd
import joblib

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

loaded = joblib.load(os.path.join(_root, 'models', 'best_model.joblib'))
model = loaded['model']
scaler = loaded['scaler']
FEATURES = loaded['features']
TARGET_COL = loaded.get('target', 'target')
BEST_THRESHOLD = float(loaded.get('threshold', 0.6))

df = pd.read_parquet(os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet'))
valid = df.dropna(subset=FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
valid = valid.sort_values(['session_key', sort_col]).reset_index(drop=True)
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)

valid = valid.dropna(subset=['ret_next'])
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date
dates = sorted(valid['date'].unique())
test_date = dates[-1]  # последний день = test
bt = valid[valid['date'] == test_date].copy()
bt = bt.sort_values(['session_key', sort_col]).reset_index(drop=True)

X_bt = scaler.transform(bt[FEATURES].fillna(0))
proba = model.predict_proba(X_bt)[:, 1]

COMMISSION_RT = 0.001  # 0.1% round-trip (0.05% one-way)
print(f'Модель: {loaded.get("model_name", "unknown")}, val_auc={loaded.get("val_auc", np.nan):.4f}, test_auc={loaded.get("test_auc", np.nan):.4f}')
print(f'Target: {TARGET_COL}, tuned threshold: {BEST_THRESHOLD:.2f}')
print(f'Бектест: только последний день ({test_date}, {len(bt):,} баров)')

Модель: LightGBM, val_auc=0.7592, test_auc=0.6277
Target: target, tuned threshold: 0.45
Бектест: только последний день (2026-02-10, 15,356 баров)


c:\Users\Iluxa\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 2. Функция бектеста

In [18]:
def backtest_pnl(proba, ret, commission_rt=0.001, threshold=0.5, session_ids=None, hold_keep_prev=True):
    """pred=1 -> long, pred=0 -> short, -1 -> HOLD.
    hold_keep_prev: при HOLD сохраняем прошлую позицию (меньше сделок). Иначе HOLD = flat.
    Комиссия 0.1% round-trip (0.05% в одну сторону)."""
    if threshold is None:
        raw = (proba >= 0.5).astype(int)
        pred = np.where(raw == 1, 1, 0)  # no HOLD
    else:
        pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i] = 1.0
            prev = 1.0
        elif pred[i] == 0:
            pos[i] = -1.0
            prev = -1.0
        else:
            pos[i] = prev if hold_keep_prev else 0.0
    
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    if session_ids is not None:
        sess_chg = np.zeros(n, dtype=bool)
        sess_chg[1:] = (session_ids[1:] != session_ids[:-1])
        pos_prev = np.where(sess_chg, 0.0, pos_prev)
    
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    n_changes = int(pos_changed.sum())
    fee_total = n_changes * (commission_rt / 2)
    
    pnl_raw = pos * ret
    pnl_net_sum = pnl_raw.sum() - fee_total

    return {'trades': n_changes, 'gross_%': pnl_raw.sum() * 100, 'net_%': pnl_net_sum * 100}

## 3. Стратегии

- **model_every_bar** — сигнал на каждом баре (proba≥0.5 → long).
- **model_tuned** — порог из 09, HOLD в зоне [0.45, 0.55].
- **model_proba_055** — long ≥0.55, short ≤0.45, HOLD в (0.45, 0.55).
- **model_wide_hold_040_060** — зона [0.40, 0.60]: long при proba≥0.60, short при ≤0.40. При HOLD **сохраняем позицию** (не выходим в flat) → меньше сделок.
- **oracle_target** — «идеальный» сигнал по target (look-ahead, в реале недостижим).

In [19]:
ret = bt['ret_next'].values
sessions = bt['session_key'].values

r1 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=None, session_ids=sessions)
r2 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=BEST_THRESHOLD, session_ids=sessions)
r3 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.55, session_ids=sessions)
r4 = backtest_pnl(proba, ret, commission_rt=COMMISSION_RT, threshold=0.60, session_ids=sessions)  # широкая HOLD [0.40, 0.60]

target_signal = bt[TARGET_COL].values
r_oracle = backtest_pnl((target_signal == 1).astype(float), ret, commission_rt=COMMISSION_RT, threshold=None, session_ids=sessions)

print('Backtest на последнем дне (комиссия 0.1% round-trip). HOLD = сохраняем позицию.')
for name, r in [('Модель (каждый бар)', r1), (f'Модель (tuned thr={BEST_THRESHOLD:.2f})', r2),
                ('Модель (proba 0.55)', r3), ('Модель (wide HOLD 0.40-0.60)', r4), ('Oracle (look-ahead)', r_oracle)]:
    print(f'  {name:35} смен={r["trades"]:>6,}  gross={r["gross_%"]:>8.2f}%  net={r["net_%"]:>8.2f}%')

Backtest на последнем дне (комиссия 0.1% round-trip). HOLD = сохраняем позицию.
  Модель (каждый бар)                 смен= 5,228  gross= 1215.95%  net=  954.55%
  Модель (tuned thr=0.45)             смен= 4,783  gross= 1187.23%  net=  948.08%
  Модель (proba 0.55)                 смен= 3,106  gross= 1156.84%  net= 1001.54%
  Модель (wide HOLD 0.40-0.60)        смен= 1,803  gross= 1064.13%  net=  973.98%
  Oracle (look-ahead)                 смен= 2,324  gross= 4664.86%  net= 4548.66%


## 4. Сводная таблица

`trades` — число смен позиции (каждая = 0.05% комиссии). При HOLD позиция сохраняется, меняется только при явном сигнале long/short.

In [20]:
summary = pd.DataFrame([
    {'strategy': 'model_every_bar', 'trades': r1['trades'], 'gross_%': r1['gross_%'], 'net_%': r1['net_%']},
    {'strategy': f'model_tuned_thr_{BEST_THRESHOLD:.2f}', 'trades': r2['trades'], 'gross_%': r2['gross_%'], 'net_%': r2['net_%']},
    {'strategy': 'model_proba_055', 'trades': r3['trades'], 'gross_%': r3['gross_%'], 'net_%': r3['net_%']},
    {'strategy': 'model_wide_hold_040_060', 'trades': r4['trades'], 'gross_%': r4['gross_%'], 'net_%': r4['net_%']},
    {'strategy': 'oracle_target', 'trades': r_oracle['trades'], 'gross_%': r_oracle['gross_%'], 'net_%': r_oracle['net_%']},
]).sort_values('net_%', ascending=False)
print(summary.to_string(index=False))

               strategy  trades     gross_%       net_%
          oracle_target    2324 4664.861231 4548.661231
        model_proba_055    3106 1156.835459 1001.535459
model_wide_hold_040_060    1803 1064.127341  973.977341
        model_every_bar    5228 1215.953439  954.553439
   model_tuned_thr_0.45    4783 1187.230455  948.080455


## 5. Интерпретация и выводы

### Результаты по стратегиям (последний день, 15 356 баров)

| Стратегия | Сделок | net_% | Комментарий |
|-----------|--------|-------|-------------|
| oracle_target | 2 324 | 4 549% | Look-ahead, недостижим в реале |
| model_proba_055 | 3 106 | **1 002%** | Максимальный net среди реалистичных |
| model_wide_hold_040_060 | **1 803** | 974% | Минимум сделок при сопоставимом net |
| model_every_bar | 5 228 | 955% | Много сделок, выше комиссии |
| model_tuned_thr_0.45 | 4 783 | 948% | Близко к every_bar по net |

### Выводы

1. **model_proba_055** даёт лучший net_% среди реализуемых стратегий (~1002%) при умеренном числе сделок (3 106).

2. **model_wide_hold_040_060** даёт наименьшее число сделок (1 803, почти в 3 раза меньше tuned) при net_% 974%. Хороший компромисс: меньше turnover и комиссий, близкий результат.

3. **Широкий HOLD с сохранением позиции** — сделок меньше, gross падает, но за счёт снижения комиссий net остаётся высоким.

4. **model_every_bar** и **model_tuned** — больше сделок и ниже net, чем у proba_055 и wide_hold.

5. **Oracle** — целевой верхний предел (~4549% net), но в реальной торговле недостижим.

### Рекомендации

- Для максимального net: **model_proba_055**.
- Для минимизации turnover при сохранении результата: **model_wide_hold_040_060**.